# 논문1 DNN 모델 + 강원도 날씨 데이터 적용 정리

이 노트북은 흩어져 있는 모델링 관련 파일을 한 번에 보기 위한 정리 파일입니다.

핵심 결론은 다음과 같습니다.

- 논문에는 **학습 완료된 모델 파일**이 있는 것이 아니라, DNN 모델 구조와 실험 방법이 설명되어 있습니다.
- 논문에서 가장 좋은 결과를 보인 모델은 **기상 변수 5개만 사용하는 DNN 회귀 모델**입니다.
- 강원도 날씨 통합 시간단위 데이터는 논문 입력 형식으로 변환해두었습니다.
- 하지만 현재 강원도 2020~2021 산불 데이터에는 `피해면적(ha)` 값이 비어 있어, 논문 DNN을 실제로 학습한 예측 결과는 아직 없습니다.
- 현재 확인 가능한 결과는 **모델 입력 데이터**와 **규칙 기반 날씨 위험 점수**입니다.

## 1. 파일 경로 정리

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path('../..').resolve()

weather_input_path = ROOT / 'data/modeling/paper1_dnn_gangwon_weather_input_from_integrated_time.csv'
daily_risk_path = ROOT / 'data/modeling/gangwon_weather_daily_model_features.csv'
pole_risk_path = ROOT / 'data/modeling/gangwon_poles_weather_model_input_2021-12-31.csv'
weights_path = ROOT / 'data/modeling/paper1_dnn_weights.npz'
prediction_path = ROOT / 'data/modeling/paper1_dnn_gangwon_weather_predictions.csv'

paths = {
    '논문 DNN 입력 데이터': weather_input_path,
    '일단위 날씨 위험 점수': daily_risk_path,
    '전신주별 날씨 입력/위험 점수': pole_risk_path,
    '논문 DNN 학습 가중치': weights_path,
    '논문 DNN 예측 결과': prediction_path,
}

for name, path in paths.items():
    print(f'{name}: {path}')
    print(f'  존재 여부: {path.exists()}')

## 2. 논문 모델 설명

논문1은 산불 예측을 **회귀 문제**로 봅니다.

예측 대상은 산불 피해면적이고, 타깃은 다음처럼 로그 변환합니다.

```text
y = ln(피해면적 + 1)
```

논문에서 가장 성능이 좋았다고 설명한 입력 조합은 기상 변수만 사용하는 `M` 셋업입니다.

```text
avg_temp
min_temp
max_temp
max_wind_speed
avg_wind
```

DNN 구조는 다음 범위에서 실험했습니다.

```text
hidden layers: 2~5개
neurons per hidden layer: 32~64개
optimizer: SGD, Adagrad, RMSprop, Adam
activation: ReLU
output activation: 없음
```

## 3. 강원도 날씨 데이터를 논문 입력 형식으로 변환한 결과

In [ ]:
weather_input = pd.read_csv(weather_input_path, encoding='utf-8-sig')

print('행/열:', weather_input.shape)
print('날짜 범위:', weather_input['date'].min(), '~', weather_input['date'].max())

paper_features = ['avg_temp', 'min_temp', 'max_temp', 'max_wind_speed', 'avg_wind']
print('필수 피처 누락:', [col for col in paper_features if col not in weather_input.columns])

weather_input.head()

In [ ]:
weather_input[paper_features].describe().round(3)

## 4. 현재 논문 DNN 예측 결과가 없는 이유

아래 두 파일이 있어야 논문 DNN의 실제 예측 결과가 있다고 볼 수 있습니다.

- `paper1_dnn_weights.npz`: 학습된 DNN 가중치
- `paper1_dnn_gangwon_weather_predictions.csv`: 학습된 DNN으로 만든 예측 결과

현재는 피해면적 타깃 데이터가 없어서 학습을 못 했기 때문에 두 파일이 없습니다.

In [ ]:
print('학습 가중치 존재:', weights_path.exists())
print('DNN 예측 결과 존재:', prediction_path.exists())

## 5. 대신 현재 확인 가능한 결과: 규칙 기반 날씨 위험 점수

`weather_risk_score_0_1`은 학습된 모델 예측값이 아닙니다.

풍속, 습도, 강수량을 조합해서 만든 참고용 날씨 위험 점수입니다.

해석:

- 1에 가까울수록 강풍, 저습, 무강수 조건이 강함
- 0에 가까울수록 날씨 조건만 보면 위험도가 낮음
- 실제 산불 발생 확률이나 피해면적 예측값은 아님

In [ ]:
daily_risk = pd.read_csv(daily_risk_path, encoding='utf-8-sig')

print('행/열:', daily_risk.shape)
print('날짜 범위:', daily_risk['날짜'].min(), '~', daily_risk['날짜'].max())
print('\n위험 점수 요약')
print(daily_risk['weather_risk_score_0_1'].describe().round(4))
print('\nfire_weather_flag 개수')
print(daily_risk['fire_weather_flag'].value_counts())

## 6. 가장 위험 점수가 높았던 날짜/기상셀

In [ ]:
risk_cols = [
    '기상셀ID', '날짜', '기후권역', '기후지형유형',
    'daily_temp_mean_C', 'daily_rain_mm', 'daily_gust_max_m_s',
    'hourly_humidity_min_pct', 'fire_weather_flag', 'weather_risk_score_0_1'
]

daily_risk.sort_values('weather_risk_score_0_1', ascending=False)[risk_cols].head(10)

현재 계산 기준 최고 위험 조건은 다음과 같습니다.

```text
기상셀ID: YS_0032
날짜: 2021-01-05
기후권역: 영서
기후지형유형: 고지·산간형
일강수량: 0.0 mm
일최대순간풍속: 18.04 m/s
시간단위 최저습도: 4.29 %
weather_risk_score_0_1: 0.9839
```

## 7. 전신주별 2021-12-31 날씨 위험 점수

In [ ]:
pole_cols = [
    'pole_id', 'lon', 'lat', '기상셀ID', '날짜', '기후권역', '기후지형유형',
    'daily_gust_max_m_s', 'hourly_humidity_min_pct',
    'fire_weather_flag', 'weather_risk_score_0_1'
]

pole_risk = pd.read_csv(pole_risk_path, encoding='utf-8-sig', usecols=pole_cols)

print('전신주 수:', len(pole_risk))
print('\n위험 점수 요약')
print(pole_risk['weather_risk_score_0_1'].describe().round(4))
print('\nfire_weather_flag 개수')
print(pole_risk['fire_weather_flag'].value_counts())

In [ ]:
pole_risk.sort_values('weather_risk_score_0_1', ascending=False).head(10)

## 8. 최종 정리

현재 상태를 정확히 구분하면 다음과 같습니다.

```text
완료된 것:
- 강원도 날씨 통합 시간단위 데이터 생성
- 논문 DNN 입력 형식으로 날씨 피처 변환
- 전신주별 날씨 격자 매칭
- 규칙 기반 날씨 위험 점수 계산
- 논문 DNN 구조를 코드로 구현

아직 안 된 것:
- 논문 DNN 실제 학습
- 논문 DNN 실제 예측 결과 생성

이유:
- 피해면적(ha) 타깃 데이터가 없어서 지도학습을 할 수 없음

다음 단계:
- 피해면적이 있는 산불 데이터 확보
- 날씨 입력 피처와 산불 피해면적 타깃 결합
- paper1_dnn_model.py train 실행
- 학습된 모델로 predict 실행
```